# Phase 2+3: 3役スキャフォールド実装 & 評価

**Phase 2**: Summarizer / Agent / Corrector の実装とテスト
**Phase 3**: test_normal 168 タスクでスキャフォールド評価

## 1. 環境セットアップ

In [ ]:
!pip install -q "pydantic==1.10.26" "fastapi==0.110.3"
!pip install -q appworld huggingface_hub openai tiktoken
!appworld install
!appworld download data

In [ ]:
import os
import json
import re
import subprocess
import time
import requests
from pathlib import Path
from datetime import datetime
from collections import Counter

In [ ]:
!nvidia-smi

## 2. Bonsai-8B 推論サーバー起動

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_DIR = Path("/content/bonsai-8b-gguf")
MODEL_DIR.mkdir(exist_ok=True)

model_path = hf_hub_download(
    repo_id="prism-ml/Bonsai-8B-gguf",
    filename="Bonsai-8B.gguf",
    local_dir=str(MODEL_DIR),
)
print(f"Model: {model_path}")

In [ ]:
!git clone https://github.com/PrismML-Eng/llama.cpp /content/llama-cpp-prism 2>/dev/null || echo "Already cloned"
%cd /content/llama-cpp-prism
!cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native 2>&1 | tail -3
!cmake --build build --config Release -j$(nproc) 2>&1 | tail -5
%cd /content

In [ ]:
!pkill -9 -f llama-server 2>/dev/null; sleep 3

server_proc = subprocess.Popen(
    [
        "/content/llama-cpp-prism/build/bin/llama-server",
        "-m", str(list(MODEL_DIR.glob("*.gguf"))[0]),
        "--host", "0.0.0.0",
        "--port", "8090",
        "-ngl", "99",
        "-c", "16384",
    ],
    stdout=open("/content/server.log", "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(20)
!cat /content/server.log | tail -5

In [ ]:
from openai import OpenAI

BONSAI_PORT = 8090
client = OpenAI(base_url=f"http://localhost:{BONSAI_PORT}/v1", api_key="dummy")

response = client.chat.completions.create(
    model="bonsai-8b",
    messages=[{"role": "user", "content": "What is 15 + 27?"}],
    temperature=0, max_tokens=200, seed=100,
)
print(f"Response: {response.choices[0].message.content}")
print("Server OK")

In [ ]:
import appworld
appworld.update_root("/content")
from appworld import AppWorld, load_task_ids
from appworld.task import Task

## 3. スキャフォールド実装

論文 Figure 1 のデータフロー:
```
observation → Summarizer(条件付き) → 圧縮履歴
           → Agent(圧縮履歴で推論) → 提案コード a_t
           → Corrector(a_t + API docs, 履歴なし) → 修正コード a_t'
           → AppWorld.execute()
```

In [ ]:
# --- トークンカウント用 ---
import tiktoken

try:
    _encoder = tiktoken.get_encoding("cl100k_base")
except Exception:
    _encoder = None

def count_tokens(text: str) -> int:
    """テキストのトークン数を推定"""
    if _encoder:
        return len(_encoder.encode(text))
    return len(text) // 4  # フォールバック: 4文字≒1トークン

def count_messages_tokens(messages: list[dict]) -> int:
    """メッセージリスト全体のトークン数を推定"""
    return sum(count_tokens(m.get("content", "")) for m in messages)

In [ ]:
# --- Summarizer システムプロンプト ---
# 論文 p.7: Tier 2 の仕様に基づく

SUMMARIZER_PROMPT = """You are a conversation summarizer for an AI coding agent.
Your task is to compress the conversation history while preserving critical information.

You MUST preserve the following verbatim (do not paraphrase):
- Authentication tokens and credentials (access tokens, API keys, session IDs)
- API endpoint names and their observed response schemas
- Error messages and their resolutions
- Pagination states and iteration progress
- Task completion status indicators
- User IDs, email addresses, and other identifiers discovered during execution

Output a concise summary that another AI agent can use to continue the task.
Start with "Summary of previous actions:" and list the key facts."""

In [ ]:
# --- Corrector システムプロンプト ---
# 論文 p.7: Tier 3 の仕様に基づく
# Corrector は履歴 h_t にアクセスしない (Eq. 3 の正則化設計)

CORRECTOR_PROMPT = """You are a code correction agent. You receive a proposed Python code snippet and relevant API documentation.
You do NOT have access to the conversation history.

Your task:
1. Classify the issue (if any) into: authentication error, API schema mismatch, wrong API name, formatting error, or no issue
2. Cite evidence from the execution output (if provided)
3. Output a corrected version of the code

Rules:
- The corrected code MUST contain exactly one ```python ... ``` code block
- The code MUST include at least one `apis.*` call
- Parameter names MUST match the API documentation exactly
- If you are uncertain about the correct parameters, query the documentation:
  `apis.api_docs.show_api_doc("app_name", "endpoint_name")`
- If the original code looks correct, return it unchanged

Output format:
Diagnosis: <one line>
```python
<corrected code>
```"""

In [ ]:
# --- Agent システムプロンプト ---

AGENT_PROMPT = """You are an AI assistant that solves tasks by writing Python code.
You have access to APIs via the `apis` object. Write code to accomplish the task.
Always wrap your code in ```python ... ``` blocks.
Keep your code concise and focused on the task.
When you are done, call `apis.supervisor.complete_task()` to finish."""

In [ ]:
# --- ヘルパー関数 ---

def call_llm(messages: list[dict], max_tokens: int = 3000) -> str:
    """Bonsai LLM 呼び出し"""
    response = client.chat.completions.create(
        model="bonsai-8b",
        messages=messages,
        temperature=0,
        max_tokens=max_tokens,
        seed=100,
    )
    return response.choices[0].message.content


def extract_code(text: str) -> str | None:
    """Python コードブロックを抽出"""
    pattern = r"```python\s*\n(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    if "apis." in text:
        return text.strip()
    return None


def extract_api_endpoints(code: str) -> list[str]:
    """コードから apis.xxx.yyy() の呼び出しを抽出"""
    pattern = r"apis\.(\w+)\.(\w+)"
    return [f"{m[0]}.{m[1]}" for m in re.findall(pattern, code)]

In [ ]:
# --- Summarizer ---
# 論文 p.7: |h_t| > 24000文字 or > 6000トークンで発動
# Bonsai 用に半減: 12000文字 / 3000トークン
# N=13 先頭メッセージ + K=3 末尾メッセージを保持

SUMMARIZER_CHAR_THRESHOLD = 12000
SUMMARIZER_TOKEN_THRESHOLD = 3000
N_FIRST = 13  # 先頭保持数
K_LAST = 3    # 末尾保持数


def should_summarize(messages: list[dict]) -> bool:
    """Summarizer を発動すべきか判定"""
    total_chars = sum(len(m.get("content", "")) for m in messages)
    total_tokens = count_messages_tokens(messages)
    return total_chars > SUMMARIZER_CHAR_THRESHOLD or total_tokens > SUMMARIZER_TOKEN_THRESHOLD


def run_summarizer(messages: list[dict]) -> list[dict]:
    """履歴を圧縮して新しいメッセージリストを返す

    N 先頭 + 中間要約 + K 末尾の構成にする
    """
    if len(messages) <= N_FIRST + K_LAST:
        return messages  # 短すぎる場合はそのまま

    first_messages = messages[:N_FIRST]
    last_messages = messages[-K_LAST:]
    middle_messages = messages[N_FIRST:-K_LAST]

    # 中間部分を要約
    middle_text = "\n".join(
        f"[{m['role']}]: {m['content'][:500]}" for m in middle_messages
    )

    summary_request = [
        {"role": "system", "content": SUMMARIZER_PROMPT},
        {"role": "user", "content": f"Summarize the following conversation segment:\n\n{middle_text}"},
    ]

    try:
        summary = call_llm(summary_request, max_tokens=1000)
    except Exception:
        return messages  # 要約失敗時は元のまま

    # 先頭 + 要約 + 末尾 を結合
    summarized = first_messages + [
        {"role": "user", "content": f"[CONTEXT SUMMARY]\n{summary}"}
    ] + last_messages

    return summarized

In [ ]:
# --- Corrector ---
# 論文 p.7, Eq. 3: π_correct(a_t' | a_t, d_t)
# 履歴 h_t にアクセスしない

def run_corrector(
    agent_code: str,
    last_execution_output: str | None = None,
) -> str:
    """Agent の提案コードを修正する

    入力: agent_code (a_t), 直前の実行結果 (失敗時のみ)
    出力: 修正済みコード (a_t')
    履歴にはアクセスしない (論文の正則化設計)
    """
    # API ドキュメント情報を構築
    endpoints = extract_api_endpoints(agent_code)
    api_info = f"Referenced API endpoints: {', '.join(endpoints)}" if endpoints else "No API endpoints detected"

    corrector_input = f"Proposed code:\n```python\n{agent_code}\n```\n\n{api_info}"

    if last_execution_output:
        corrector_input += f"\n\nLast execution output:\n```\n{last_execution_output[:1000]}\n```"

    corrector_messages = [
        {"role": "system", "content": CORRECTOR_PROMPT},
        {"role": "user", "content": corrector_input},
    ]

    try:
        corrector_output = call_llm(corrector_messages, max_tokens=2000)
        corrected_code = extract_code(corrector_output)
        if corrected_code:
            return corrected_code
    except Exception:
        pass

    # フォールバック: Corrector 失敗時は Agent 出力をそのまま使う
    return agent_code

In [ ]:
# --- スキャフォールド統合エージェント ---

def run_scaffold_task(task_id: str, max_steps: int = 40) -> dict:
    """3役スキャフォールドで 1 タスクを実行

    各ターン:
    1. Summarizer (条件付き): 履歴が閾値超えたら圧縮
    2. Agent: 圧縮済み履歴でコード生成
    3. Corrector: Agent 出力を履歴なしで修正
    """
    result = {
        "task_id": task_id,
        "success": False,
        "steps": 0,
        "turns": [],
        "wall_time": 0,
        "error": None,
        "summarizer_invocations": 0,
        "corrector_changes": 0,
    }
    t0 = time.time()

    try:
        with AppWorld(task_id=task_id, experiment_name="phase3_scaffold") as world:
            messages = [
                {"role": "system", "content": AGENT_PROMPT},
                {"role": "user", "content": (
                    f"Task: {world.task.instruction}\n\n"
                    f"Supervisor: {world.task.supervisor}\n\n"
                    f"Available apps: {list(world.task.app_descriptions.keys())}"
                )},
            ]

            last_execution_output = None

            for step in range(max_steps):
                turn_t0 = time.time()
                turn_log = {"step": step, "summarizer_used": False, "corrector_changed": False}

                # --- Tier 1: Summarizer (条件付き) ---
                if should_summarize(messages):
                    messages = run_summarizer(messages)
                    turn_log["summarizer_used"] = True
                    result["summarizer_invocations"] += 1

                # --- Tier 2: Agent ---
                try:
                    agent_output = call_llm(messages)
                except Exception as llm_err:
                    result["error"] = str(llm_err)
                    result["steps"] = step
                    break

                agent_code = extract_code(agent_output)
                turn_log["agent_output"] = agent_output
                turn_log["agent_code_extracted"] = agent_code is not None

                if agent_code is None:
                    turn_log["observation"] = "ERROR: No code block found in Agent output"
                    messages.append({"role": "assistant", "content": agent_output})
                    messages.append({"role": "user", "content": turn_log["observation"]})
                    turn_log["turn_time"] = time.time() - turn_t0
                    result["turns"].append(turn_log)
                    continue

                # --- Tier 3: Corrector ---
                corrected_code = run_corrector(agent_code, last_execution_output)
                turn_log["corrector_changed"] = (corrected_code != agent_code)
                if turn_log["corrector_changed"]:
                    result["corrector_changes"] += 1

                final_code = corrected_code

                # --- AppWorld 実行 ---
                try:
                    output = world.execute(final_code)
                    turn_log["observation"] = output
                    last_execution_output = output
                except Exception as e:
                    output = f"Execution error: {e}"
                    turn_log["observation"] = output
                    last_execution_output = output

                # 履歴には Agent の出力を追加 (Corrector の修正は履歴に入れない)
                messages.append({"role": "assistant", "content": agent_output})
                messages.append({"role": "user", "content": f"Output:\n```\n{output}\n```"})

                turn_log["turn_time"] = time.time() - turn_t0
                result["turns"].append(turn_log)
                result["steps"] = step + 1

                if world.task_completed():
                    result["success"] = True
                    break

    except Exception as e:
        result["error"] = str(e)

    result["wall_time"] = time.time() - t0
    return result

## 4. Phase 2: スキャフォールドテスト (1-2 タスク)

In [ ]:
# dev の難易度 1 タスクで動作確認
dev_task_ids = load_task_ids("dev")

difficulty_1_dev = []
for tid in dev_task_ids:
    task = Task.load(task_id=tid)
    if task.ground_truth.metadata["difficulty"] == 1:
        difficulty_1_dev.append(tid)

test_tasks = difficulty_1_dev[:2]
print(f"Testing scaffold on {len(test_tasks)} tasks: {test_tasks}")

for tid in test_tasks:
    print(f"\n{'='*60}")
    print(f"Task: {tid}")
    result = run_scaffold_task(tid, max_steps=10)
    print(f"Success: {result['success']}")
    print(f"Steps: {result['steps']}")
    print(f"Wall time: {result['wall_time']:.1f}s")
    print(f"Summarizer invocations: {result['summarizer_invocations']}")
    print(f"Corrector changes: {result['corrector_changes']}")
    print(f"Error: {result['error']}")
    for turn in result["turns"]:
        s = turn["step"]
        summ = " [SUMMARIZED]" if turn.get("summarizer_used") else ""
        corr = " [CORRECTED]" if turn.get("corrector_changed") else ""
        t = turn.get("turn_time", 0)
        print(f"  Step {s}: {t:.1f}s{summ}{corr}")

## 5. Phase 3: スキャフォールド評価 (test_normal 168 タスク)

**所要時間見込み**: ベースラインの 2-3 倍 → 約 5-9 時間

In [ ]:
test_task_ids = load_task_ids("test_normal")
print(f"test_normal: {len(test_task_ids)} tasks")

RESULTS_DIR = Path("/content/phase3_results")
RESULTS_DIR.mkdir(exist_ok=True)

checkpoint_file = RESULTS_DIR / "phase3_scaffold.jsonl"

all_results = []
completed_ids = set()
if checkpoint_file.exists():
    with open(checkpoint_file) as f:
        for line in f:
            r = json.loads(line)
            completed_ids.add(r["task_id"])
            all_results.append(r)
    print(f"Resuming: {len(completed_ids)}/{len(test_task_ids)} tasks already completed")

start_time = time.time()

for i, task_id in enumerate(test_task_ids):
    if task_id in completed_ids:
        continue

    done = len(completed_ids) + (len(all_results) - len(completed_ids))
    if done > len(completed_ids):
        elapsed = time.time() - start_time
        avg = elapsed / (done - len(completed_ids))
        remaining = len(test_task_ids) - done
        eta_min = avg * remaining / 60
        eta_str = f", ETA: {eta_min:.0f}min"
    else:
        eta_str = ""

    print(f"[{i+1}/{len(test_task_ids)}] {task_id}{eta_str}")
    result = run_scaffold_task(task_id, max_steps=40)
    all_results.append(result)
    completed_ids.add(task_id)

    with open(checkpoint_file, "a") as f:
        f.write(json.dumps(result, ensure_ascii=False, default=str) + "\n")

    status = "OK" if result["success"] else "FAIL"
    summ = f" summ={result['summarizer_invocations']}"
    corr = f" corr={result['corrector_changes']}"
    err = f" [{result['error'][:50]}]" if result["error"] else ""
    print(f"  {status} steps={result['steps']} time={result['wall_time']:.1f}s{summ}{corr}{err}")

total_time = (time.time() - start_time) / 60
print(f"\nTotal time: {total_time:.1f} min")

## 6. 結果分析 (Before/After 比較)

In [ ]:
print("=" * 60)
print("PHASE 3 SCAFFOLD RESULTS")
print("=" * 60)

successes = sum(1 for r in all_results if r["success"])
total = len(all_results)
errors = sum(1 for r in all_results if r["error"])
times = [r["wall_time"] for r in all_results]
steps = [r["steps"] for r in all_results]
summ_total = sum(r["summarizer_invocations"] for r in all_results)
corr_total = sum(r["corrector_changes"] for r in all_results)

print(f"Tasks: {total}")
print(f"Success: {successes}/{total} ({successes/total*100:.1f}%)")
print(f"Errors: {errors}/{total}")
print(f"Wall time: mean={sum(times)/len(times):.1f}s, total={sum(times)/60:.1f}min")
print(f"Summarizer invocations: {summ_total}")
print(f"Corrector changes: {corr_total}")

In [ ]:
# --- 難易度別 ---
print("\n=== Results by Difficulty ===")
print(f"{'Difficulty':<12} {'Baseline':<12} {'Scaffold':<12} {'Improvement'}")

# Phase 1 ベースラインは全て 0% だったので直接記載
baseline_by_d = {1: 0, 2: 0, 3: 0}
scaffold_by_d = {}

for r in all_results:
    task = Task.load(task_id=r["task_id"])
    d = task.ground_truth.metadata["difficulty"]
    if d not in scaffold_by_d:
        scaffold_by_d[d] = {"total": 0, "success": 0}
    scaffold_by_d[d]["total"] += 1
    if r["success"]:
        scaffold_by_d[d]["success"] += 1

for d in sorted(scaffold_by_d):
    info = scaffold_by_d[d]
    s_rate = info["success"] / info["total"] * 100 if info["total"] > 0 else 0
    b_rate = 0.0  # Phase 1 は全て 0%
    improvement = s_rate - b_rate
    print(f"  {d:<10} {b_rate:<10.1f}% {s_rate:<10.1f}% +{improvement:.1f}pp")

In [ ]:
# --- 論文比較 (参考値) ---
print("\n=== Comparison with Paper (reference only) ===")
print("Model                    Baseline  Scaffold")
print(f"Bonsai-8B (1-bit, ours)  0.0%      {successes/total*100:.1f}%")
print(f"Qwen3-8B FP16 (paper)    5.4%      8.9%")
print(f"Qwen3-8B AWQ (paper)     3.0%      5.9%")

In [ ]:
# --- スキャフォールドの効果分析 ---
print("\n=== Scaffold Effect Analysis ===")
turns_with_correction = sum(
    1 for r in all_results for t in r["turns"] if t.get("corrector_changed")
)
total_turns = sum(len(r["turns"]) for r in all_results)
print(f"Turns with Corrector changes: {turns_with_correction}/{total_turns} "
      f"({turns_with_correction/total_turns*100:.1f}%)" if total_turns > 0 else "No turns")

tasks_with_summarizer = sum(1 for r in all_results if r["summarizer_invocations"] > 0)
print(f"Tasks using Summarizer: {tasks_with_summarizer}/{total}")

In [ ]:
# --- エラーパターン ---
print("\n=== Error Patterns ===")
error_types = Counter()
for r in all_results:
    if r["error"]:
        if "exceeds the available" in r["error"]:
            error_types["context_overflow"] += 1
        elif "Connection" in r["error"]:
            error_types["connection_error"] += 1
        else:
            error_types["other"] += 1

for etype, count in error_types.most_common():
    print(f"  {etype}: {count}")

# Phase 1 との比較
print(f"\nContext overflow: Phase 1 = 22, Phase 3 = {error_types.get('context_overflow', 0)}")

## 7. レポート

In [ ]:
report = {
    "timestamp": datetime.now().isoformat(),
    "phase": "3-scaffold",
    "inference_backend": "llama.cpp (PrismML fork)",
    "gpu": "NVIDIA A100-SXM4-40GB",
    "context_length": 16384,
    "dataset": "test_normal",
    "total_tasks": total,
    "successes": successes,
    "success_rate": successes / total if total > 0 else 0,
    "errors": errors,
    "mean_wall_time_seconds": sum(times) / len(times) if times else 0,
    "total_wall_time_minutes": sum(times) / 60 if times else 0,
    "summarizer_invocations_total": summ_total,
    "corrector_changes_total": corr_total,
    "by_difficulty": {
        str(d): {"total": info["total"], "success": info["success"]}
        for d, info in sorted(scaffold_by_d.items())
    },
    "error_patterns": dict(error_types),
    "baseline_comparison": {
        "baseline_success_rate": 0.0,
        "scaffold_success_rate": successes / total if total > 0 else 0,
        "absolute_improvement": successes / total if total > 0 else 0,
        "baseline_context_overflow": 22,
        "scaffold_context_overflow": error_types.get("context_overflow", 0),
    },
}

report_file = RESULTS_DIR / "phase3_report.json"
with open(report_file, "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2, ensure_ascii=False))